In [ ]:
import os
import pickle
import torch
import random
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader

# Mapping for Amino Acids
AA_TO_INDEX = {
    'A': 0, 'C': 1, 'D': 2, 'E': 3, 'F': 4,
    'G': 5, 'H': 6, 'I': 7, 'K': 8, 'L': 9,
    'M': 10, 'N': 11, 'P': 12, 'Q': 13, 'R': 14,
    'S': 15, 'T': 16, 'V': 17, 'W': 18, 'Y': 19,
    '-': 20, 'X': 20
}

# 19-dimensional fixed structural features
DEFAULT_STRUCT_NAMES = [
    "plddt_center","plddt_neigh_mean","plddt_neigh_min","plddt_neigh_max",
    "msf_center","msf_neigh_mean", "lrco", "d_volume","d_hydropathy","d_charge",
    "ddg_fold", "sin_phi","cos_phi","sin_psi","cos_psi",
    "gly_switch","pro_switch", "delta_psic","shannon"
]
S_DIM = len(DEFAULT_STRUCT_NAMES) 

class VoxelDataset(Dataset):
    def __init__(self, df, voxel_cache_dir, struct_cache_dir, aug=False):
        self.df = df.reset_index(drop=True)
        self.voxel_cache_dir = voxel_cache_dir
        self.struct_cache_dir = struct_cache_dir
        self.aug = aug

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        uid, mut_pos, wt, mut, label = row["UniProtID"], row["MutPos"], row["WT"], row["Mut"], row["Label"]

        # 1. Load Voxel Data (63 channels)
        voxel_path = os.path.join(self.voxel_cache_dir, f"{uid}_{mut_pos}.pkl")
        with open(voxel_path, "rb") as f:
            feature = pickle.load(f)["feature"]
        
        feature_tensor = torch.from_numpy(feature).permute(0, 4, 1, 2, 3).float().squeeze(0)

        # 2. Load Structural Descriptor (S)
        key_S = f"{uid}_{int(mut_pos)}_{wt.upper()}{mut.upper()}"
        struct_path = os.path.join(self.struct_cache_dir, f"{key_S}.pkl")
        
        if os.path.exists(struct_path):
            with open(struct_path, "rb") as f:
                S = pickle.load(f)["S"]
        else:
            S = np.zeros(S_DIM, dtype=np.float32)

        # Dimension defense (Padding/Truncation)
        if S.shape[0] != S_DIM:
            S = np.pad(S, (0, S_DIM - S.shape[0])) if S.shape[0] < S_DIM else S[:S_DIM]

        S_tensor = torch.nan_to_num(torch.from_numpy(np.asarray(S, dtype=np.float32)), nan=0.0)
        ref_idx = torch.tensor(AA_TO_INDEX.get(str(wt), 20), dtype=torch.long)
        mut_idx = torch.tensor(AA_TO_INDEX.get(str(mut), 20), dtype=torch.long)

        return feature_tensor, S_tensor, ref_idx, mut_idx, torch.tensor(label).float()

class MSADataset(Dataset):
    def __init__(self, df, msa_dict_path, max_depth=80, win_size=61, aug=False):
        with open(msa_dict_path, "rb") as f:
            self.msa_dict = pickle.load(f)
        self.df, self.max_depth, self.win_size, self.aug = df, max_depth, win_size, aug
        self.half_win = win_size // 2
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        uid, mut_pos, label, mut = row["UniProtID"], int(row["MutPos"]) - 1, float(row["Label"]), row["Mut"].upper()
        
        msa_seqs = [seq for _, seq in self.msa_dict[uid][:self.max_depth]]
        query_seq = msa_seqs[0]
        mut_seq = list(query_seq)
        if 0 <= mut_pos < len(mut_seq): mut_seq[mut_pos] = mut

        seqs_to_use = [mut_seq, list(query_seq)] + [list(s) for s in msa_seqs[1:self.max_depth-2]]
        
        centered_msa = []
        for seq in seqs_to_use:
            window = [AA_TO_INDEX.get(seq[mut_pos - self.half_win + i] if 0 <= mut_pos - self.half_win + i < len(seq) else '-', 20) for i in range(self.win_size)]
            centered_msa.append(window)

        while len(centered_msa) < self.max_depth: centered_msa.append([20] * self.win_size)
        return {"msa": torch.tensor(centered_msa[:self.max_depth]).transpose(0, 1), "label": torch.tensor(label).float()}

class MultimodalDataset(Dataset):
    def __init__(self, df, voxel_cache_dir, struct_cache_dir, msa_dict_path):
        self.df = df.reset_index(drop=True)
        self.voxel_dataset = VoxelDataset(df, voxel_cache_dir, struct_cache_dir)
        self.msa_dataset = MSADataset(df, msa_dict_path)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        voxel_feat, s_tensor, ref_idx, mut_idx, label = self.voxel_dataset[idx]
        msa_tensor = self.msa_dataset[idx]["msa"]
        return {"voxel": voxel_feat, "s_tensor": s_tensor, "ref_idx": ref_idx, "mut_idx": mut_idx, "msa": msa_tensor, "label": label}

In [ ]:
from Models import EvoStructCLIP
from sklearn.model_selection import train_test_split

# 1. Config Paths
DATA_PATH = "/mnt/c/Users/Kunny/Research/Project/BiConVarNet/FGFR/KCNQ4_formatted.tsv"
VOXEL_DIR = "/mnt/e/CAGI_data/voxel_cache_4_KCNQ4_noRSA_2"
STRUCT_DIR = "/mnt/c/Users/Kunny/Research/Dataset/Missense_Variant_dataset/struct_feature_cache_KCNQ"
MSA_PATH = "/mnt/e/CAGI_data/msa_dict_valid.pkl"
MODEL_WEIGHTS = "/mnt/e/CAGI_data/best_model_250917_clip_epoch81.pth"

# 2. Data Loading
df = pd.read_csv(DATA_PATH, sep="\t")
df_train, df_val = train_test_split(df, test_size=0.05, random_state=42) 

train_loader = DataLoader(MultimodalDataset(df_train, VOXEL_DIR, STRUCT_DIR, MSA_PATH), batch_size=64, shuffle=False)
val_loader = DataLoader(MultimodalDataset(df_val, VOXEL_DIR, STRUCT_DIR, MSA_PATH), batch_size=32, shuffle=False)

# 3. Model Setup & Feature Collection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EvoStructCLIP(voxel_ch=46, mb_layers=6, embed_dim=128, use_concat=True).to(device)
model.load_state_dict(torch.load(MODEL_WEIGHTS))
model.eval()

@torch.no_grad()
def collect_meta_features(model, loader, device):
    X_list, y_list = [], []
    for batch in loader:
        out = model(batch["voxel"].to(device), batch["ref_idx"].to(device), batch["mut_idx"].to(device), batch["msa"].to(device))
        # Concatenate Voxel (128) + MSA (128) = 256 dimensions
        fused = np.concatenate([out["voxel_feat"].cpu().numpy(), out["msa_feat"].cpu().numpy()], axis=1)
        S = batch["s_tensor"].cpu().numpy()
        # Final Feature: S (19) + Fused (256) = 275 dimensions
        X_list.append(np.concatenate([S, fused], axis=1))
        y_list.append(batch["label"].view(-1).cpu().numpy())
    return np.concatenate(X_list, axis=0), np.concatenate(y_list, axis=0)

X_tr, y_tr = collect_meta_features(model, train_loader, device)
X_va, y_va = collect_meta_features(model, val_loader, device)

del model
torch.cuda.empty_cache()

In [ ]:
import joblib
import numpy as np
from scipy.stats import pearsonr
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, RandomizedSearchCV
from sklearn.metrics import make_scorer

# Define Pearson r Scorer for Optimization
pearson_scorer = make_scorer(lambda y, yhat: pearsonr(y, yhat)[0])

# --- 1. Hyperparameter Optimization ---
print("[*] Starting Random Forest Hyperparameter Search...")
rf_base = RandomForestRegressor(random_state=42, n_jobs=-1)

rf_param_dist = {
    "n_estimators": [200, 500, 1000, 2000],
    "max_depth": [None, 5, 10, 20, 40],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2", None],
    "bootstrap": [True, False],
}

rf_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=rf_param_dist,
    n_iter=30,
    scoring=pearson_scorer,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1,
    verbose=1,
    random_state=42
)
rf_search.fit(X_tr, y_tr)
rf_best_params = rf_search.best_params_

print(f"[+] Best RF CV Pearson r: {rf_search.best_score_:.4f}")
print(f"[+] Best RF Params: {rf_best_params}")

# --- 2. Train CV Ensemble (10-Fold) ---
def train_rf_ensemble(X, y, params, n_splits=10, seed=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    models = []
    print(f"[*] Training {n_splits}-fold RF Ensemble...")
    for i, (train_idx, _) in enumerate(kf.split(X)):
        model = RandomForestRegressor(random_state=seed, n_jobs=-1, **params)
        model.fit(X[train_idx], y[train_idx])
        models.append(model)
    return models

rf_ensemble = train_rf_ensemble(X_tr, y_tr, rf_best_params)
joblib.dump(rf_ensemble, "best_rf_ensemble.pkl")
print("[+] RF Ensemble saved to: best_rf_ensemble.pkl")

# --- 3. Inference & Validation ---
def predict_with_ensemble(models, X):
    preds_all = np.stack([m.predict(X) for m in models], axis=0)
    return preds_all.mean(axis=0), preds_all.std(axis=0) + 1e-6

rf_mean, rf_std = predict_with_ensemble(rf_ensemble, X_va)
r_rf, _ = pearsonr(y_va, rf_mean)

print(f"\n[RF Results] Pearson r: {r_rf:.4f}")
print(f"  - Mean Predictions (First 5): {rf_mean[:5]}")
print(f"  - Std (Uncertainty): {rf_std[:5]}")

In [ ]:
from xgboost import XGBRegressor

# --- 1. Hyperparameter Optimization ---
print("\n[*] Starting XGBoost Hyperparameter Search (GPU-accelerated)...")
xgb_base = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    device="cuda",
    random_state=42,
    n_jobs=1
)

xgb_param_dist = {
    "n_estimators": [600, 900, 1200, 1600, 2000],
    "learning_rate": [0.03, 0.05, 0.08],
    "max_depth": [4, 5, 6],
    "min_child_weight": [2, 5, 10],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.7, 0.9],
    "reg_lambda": [0.1, 1.0, 5.0, 30.0],
    "reg_alpha": [0.0, 0.1, 1.0],
    "grow_policy": ["lossguide"],
    "max_leaves": [31, 63, 127]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=xgb_param_dist,
    n_iter=20,
    scoring=pearson_scorer,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=1,
    verbose=1,
    random_state=42
)
xgb_search.fit(X_tr, y_tr)

# Format best params (ensure correct types for XGBoost)
xgb_best_params = {k: (int(v) if isinstance(v, np.integer) else float(v) if isinstance(v, np.floating) else v)
                   for k, v in xgb_search.best_params_.items()}

print(f"[+] Best XGB CV Pearson r: {xgb_search.best_score_:.4f}")
print(f"[+] Best XGB Params: {xgb_best_params}")

# --- 2. Train CV Ensemble (10-Fold) ---
def train_xgb_ensemble(X, y, params, n_splits=10, seed=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    models = []
    print(f"[*] Training {n_splits}-fold XGB Ensemble...")
    for train_idx, _ in kf.split(X):
        model = XGBRegressor(
            objective="reg:squarederror",
            tree_method="hist",
            device="cuda",
            random_state=seed,
            **params
        )
        model.fit(X[train_idx], y[train_idx], verbose=False)
        models.append(model)
    return models

xgb_ensemble = train_xgb_ensemble(X_tr, y_tr, xgb_best_params)
joblib.dump(xgb_ensemble, "best_xgb_ensemble.pkl")
print("[+] XGB Ensemble saved to: best_xgb_ensemble.pkl")

# --- 3. Inference & Validation ---
xgb_mean, xgb_std = predict_with_ensemble(xgb_ensemble, X_va)
r_xgb, _ = pearsonr(y_va, xgb_mean)

print(f"\n[XGB Results] Pearson r: {r_xgb:.4f}")
print(f"  - Mean Predictions (First 5): {xgb_mean[:5]}")
print(f"  - Std (Uncertainty): {xgb_std[:5]}")

print("\n" + "="*50)
print(f"  Final Comparison (Pearson r)")
print(f"  - Random Forest: {r_rf:.4f}")
print(f"  - XGBoost:       {r_xgb:.4f}")
print("="*50)